# Google Cloud Run 챗봇 배포 실습 (ADC 키리스 아키텍처)

이 노트북은 **Google Cloud Run(완전 관리형 서버리스 컨테이너)** 환경에 **ADC(애플리케이션 기본 사용자 인증 정보)**를 활용한 Gemini 3.8 Flash & 3.7 Flash 웹 챗봇을 빌드하고 배포하는 실습 과정입니다.

### 🌟 ADC(Application Default Credentials) 방식의 장점 (`capture.png` 참조)
1. **완전한 키리스(Keyless) 보안**: 소스코드, 환경변수, Secret Manager 어디에도 API 키를 직접 저장하지 않습니다.
2. **인프라 런타임 ID 자동 활용**: Cloud Run의 기본 Compute 서비스 계정 identity를 통해 Vertex AI / Agent Platform에 접근합니다.
3. **Google 관리형 자동 토큰 순환**: 토큰 만료 및 순환(Rotation) 관리가 100% 자동화됩니다.
4. **엔터프라이즈 IAM 권한 통제**: `roles/aiplatform.user` 역할 하나로 세밀한 모델 접근 제어가 가능합니다.

---

## 1. 사전 환경 확인 및 GCP 프로젝트 설정

현재 로그인된 계정과 활성 프로젝트를 확인합니다.

In [ ]:
import sys
import subprocess
import json
IS_WIN = sys.platform == "win32"

PROJECT_ID = "iceu-songpa21"
REGION = "us-central1"
SERVICE_NAME = "gemini-chatbot-adc"

# 1. gcloud 프로젝트 설정
subprocess.run(["gcloud", "config", "set", "project", PROJECT_ID], check=True, shell=IS_WIN)
print(f"✅ 활성 프로젝트 설정 완료: {PROJECT_ID}")

# 2. 인증 계정 확인
account = subprocess.check_output(["gcloud", "config", "get-value", "account"], shell=IS_WIN).decode().strip()
print(f"👤 현재 로그인 계정: {account}")

## 2. ADC(애플리케이션 기본 사용자 인증 정보) 확인 및 로그인

로컬 환경에서 Vertex AI 모델을 호출할 수 있도록 ADC 자격 증명을 확인합니다.

In [ ]:
# ADC 토큰 존재 여부 확인
adc_check = subprocess.run(
    ["gcloud", "auth", "application-default", "print-access-token"],
    capture_output=True, text=True, shell=IS_WIN
)

if adc_check.returncode == 0:
    print("✅ 로컬 ADC 자격 증명이 이미 설정되어 있습니다!")
else:
    print("ℹ️ 로컬 ADC 자격 증명이 없습니다. 아래 명령어로 로그인을 진행하세요:")
    print("   gcloud auth application-default login")
    print("   gcloud auth application-default set-quota-project " + PROJECT_ID)

## 3. 필수 Google Cloud API 활성화

Cloud Run 및 Vertex AI Model API에 필요한 API들을 활성화합니다:
- `aiplatform.googleapis.com`: Google Cloud Model API / Vertex AI (capture.png 명시)
- `run.googleapis.com`: Cloud Run 서비스
- `cloudbuild.googleapis.com`: 소스코드 기반 자동 컨테이너 이미지 빌드
- `artifactregistry.googleapis.com`: 빌드된 컨테이너 이미지 저장소

In [ ]:
apis = [
    "aiplatform.googleapis.com",
    "run.googleapis.com",
    "cloudbuild.googleapis.com",
    "artifactregistry.googleapis.com"
]
print(f"⏳ 필수 API 활성화 중: {', '.join(apis)}")
subprocess.run(["gcloud", "services", "enable", *apis], check=True, shell=IS_WIN)
print("✅ 필수 API 활성화 완료!")

## 4. 서비스 계정 IAM 권한 부여 (ADC 활성화)

Cloud Run의 기본 Compute 서비스 계정에 `roles/aiplatform.user` 권한을 부여하여 키리스 ADC 통신을 가능하게 합니다.

In [ ]:
# 프로젝트 번호 조회
pnum = subprocess.check_output([
    "gcloud", "projects", "describe", PROJECT_ID,
    "--format=value(projectNumber)"
], shell=IS_WIN).decode().strip()

compute_sa = f"{pnum}-compute@developer.gserviceaccount.com"
print(f"🔍 Compute Engine 기본 서비스 계정: {compute_sa}")

# roles/aiplatform.user 바인딩
subprocess.run([
    "gcloud", "projects", "add-iam-policy-binding", PROJECT_ID,
    f"--member=serviceAccount:{compute_sa}",
    "--role=roles/aiplatform.user"
], shell=IS_WIN)
print("✅ 서비스 계정에 Vertex AI User 권한 바인딩 완료!")

## 5. Google Cloud Run 컨테이너 빌드 및 배포

`--source .` 옵션을 통해 현재 디렉터리의 소스코드를 Cloud Build에서 컨테이너로 빌드하고 배포합니다.
**비밀키(--set-secrets) 없이 환경변수만 주입**합니다.

In [ ]:
deploy_cmd = [
    "gcloud", "run", "deploy", SERVICE_NAME,
    "--source=.",
    f"--project={PROJECT_ID}",
    f"--region={REGION}",
    "--platform=managed",
    "--allow-unauthenticated",
    "--min-instances=0",
    "--max-instances=5",
    "--memory=1Gi",
    "--cpu=1",
    "--timeout=120",
    f"--set-env-vars=GOOGLE_CLOUD_PROJECT={PROJECT_ID},GOOGLE_CLOUD_LOCATION={REGION},GOOGLE_GENAI_USE_VERTEXAI=true"
]

print(f"📦 Cloud Run 배포 시작: {SERVICE_NAME}")
subprocess.run(deploy_cmd, check=True, shell=IS_WIN)
print("🎉 배포 성공!")

## 6. 배포된 서비스 URL 및 헬스체크 확인

In [ ]:
import urllib.request

url_output = subprocess.check_output([
    "gcloud", "run", "services", "describe", SERVICE_NAME,
    f"--project={PROJECT_ID}",
    f"--region={REGION}",
    "--format=value(status.url)"
], shell=IS_WIN).decode().strip()

print(f"🌐 공식 서비스 URL: {url_output}")

# 헬스체크 호출
try:
    with urllib.request.urlopen(f"{url_output}/health", timeout=10) as res:
        status_data = json.loads(res.read().decode())
        print("🩺 헬스체크 응답:", json.dumps(status_data, indent=2, ensure_ascii=False))
except Exception as e:
    print("헬스체크 호출 실패:", e)